# 01 · Baseline model comparison

**Question:** Which standard classifier performs best on the seven original Titanic passenger features?

We compare Logistic Regression, Random Forest, Extra Trees and Gradient Boosting under the **same five stratified cross-validation folds**. Imputation and encoding are fitted *inside* each training fold to avoid leakage. Logistic Regression also uses numeric scaling.

**Original experiment:** `05_model_comparison_04features.ipynb`.

## 1. Load the data

Download the Titanic `train.csv` and `test.csv` into `data/` as described in the README. The competition test set has no `Survived` labels and is not used for model selection.

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
from pathlib import Path

# Works when running Jupyter from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_dir = next(
    (folder for folder in candidates if (folder / 'data' / 'train.csv').exists()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        'data/train.csv not found. Download train.csv and test.csv from '
        'https://www.kaggle.com/competitions/titanic/data and put them in data/.'
    )

In [ ]:
train = pd.read_csv(project_dir / 'data' / 'train.csv')
test = pd.read_csv(project_dir / 'data' / 'test.csv')

features = ['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked']
X = train[features]
y = train['Survived']
X_test = test[features]
print(f'Training: {len(train)} passengers | Kaggle test: {len(test)} passengers')

## 2. Define preprocessing and models

The same original feature set is used for every model. Hyperparameters follow the original experiment rather than a new tuning search.

In [ ]:
def create_preprocessor(scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ('numeric', Pipeline(numeric_steps), ['Age', 'Fare', 'SibSp', 'Parch']),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]), ['Pclass', 'Sex', 'Embarked']),
        ],
        sparse_threshold=0.0,
    )


models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=5, random_state=42,
    ),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=200, max_depth=5, random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def make_pipeline(name):
    return Pipeline([
        ('preprocessing', create_preprocessor(
            scale_numeric=(name == 'Logistic Regression')
        )),
        ('classifier', clone(models[name])),
    ])

## 3. Compare five-fold CV accuracy

In [ ]:
results = []
for name in models:
    scores = cross_val_score(
        make_pipeline(name), X, y, cv=cv, scoring='accuracy'
    )
    results.append({
        'Model': name,
        'Mean CV accuracy': scores.mean(),
        'Fold std': scores.std(),
        'Fold scores': scores.round(4).tolist(),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values('Mean CV accuracy', ascending=False)
    .reset_index(drop=True)
)
display(results_df)
best_name = results_df.loc[0, 'Model']
print(f'Best mean CV accuracy: {best_name}')

**Recorded original results:** Gradient Boosting **0.8272**, Random Forest **0.8271**, Extra Trees **0.8137**, Logistic Regression **0.7969** mean accuracy. The difference between the top two was very small; the CV comparison alone does not establish a meaningful superiority.

## 4. Fit the selected model and create a Kaggle submission

The best-CV pipeline is refitted on **all labelled rows**. The resulting CSV can be submitted on Kaggle. This is the run that produced `submission_cv_best_v3.csv` in the original experiment.

In [ ]:
best_pipeline = make_pipeline(best_name)
best_pipeline.fit(X, y)
predictions = best_pipeline.predict(X_test).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions,
})
assert len(submission) == len(test)
assert submission['PassengerId'].is_unique
assert submission['Survived'].isin([0, 1]).all()
assert not submission.isna().any().any()

submission_dir = project_dir / 'submissions'
submission_dir.mkdir(exist_ok=True)
path = submission_dir / 'submission_cv_best_v3.csv'
submission.to_csv(path, index=False)
print(f'Saved {path.relative_to(project_dir)}')
display(submission.head())

### Outcome

- **Kaggle public leaderboard accuracy (original submission): 0.79665.** This is a historical score, not calculated by this notebook (Kaggle holds the test labels).
- The simple Gradient Boosting baseline was the strongest *observed public-leaderboard submission* among those shared for the project.
- Next: test whether adding domain-inspired features is a reliable improvement.